# Phishing URL Detector using Machine Learning
**IBM SkillsBuild Cyber Security Bootcamp 2026 (BharatCares) - Project Submission**
**Author:** Prince Acharya (IILM University, B.Tech CSE)

Phishing sites imitate trusted websites to steal credentials. Blocklists are always a step behind, because a new phishing site can go live and be abandoned within hours.
This project trains machine-learning models that look **only at the structure of a URL** (length, special characters, number of dots, path/query layout and so on) and decide whether it is *phishing* or *legitimate*, with no page download and no network lookup.

**Pipeline:** load data -> clean -> explore -> train 4 models -> evaluate -> feature importance -> ablation (URL-only vs. full features) -> save model -> live prediction on raw URLs.

**Dataset:** Vrbancic et al., *Phishing-Dataset* (`dataset_small.csv`, 58,645 URLs, 111 features, balanced). https://github.com/GregaVrbancic/Phishing-Dataset

In [1]:
import os, json, re, time, warnings, urllib.request
from urllib.parse import urlsplit
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg") if os.environ.get("NB_HEADLESS") else None
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix, ConfusionMatrixDisplay,
                             classification_report, roc_curve)

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
SEED = 42
np.random.seed(SEED)
for d in ("data", "images", "models", "results"):
    os.makedirs(d, exist_ok=True)

## 1. Load the dataset
The notebook downloads the CSV from GitHub the first time it runs (about 16 MB) and reuses the local copy afterwards.

In [2]:
DATA_URL = "https://raw.githubusercontent.com/GregaVrbancic/Phishing-Dataset/master/dataset_small.csv"
DATA_PATH = "data/dataset_small.csv"
if not os.path.exists(DATA_PATH):
    print("Downloading dataset ...")
    urllib.request.urlretrieve(DATA_URL, DATA_PATH)

df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
df.head()

Shape: (58645, 112)


,qty_dot_url,qty_hyphen_url,qty_underline_url,qty_slash_url,qty_questionmark_url,qty_equal_url,qty_at_url,qty_and_url,qty_exclamation_url,qty_space_url,...,qty_ip_resolved,qty_nameservers,qty_mx_servers,ttl_hostname,tls_ssl_certificate,qty_redirects,url_google_index,domain_google_index,url_shortened,phishing
0,2,0,0,0,0,0,0,0,0,0,...,1,4,2,3598,0,0,0,0,0,0
1,4,0,0,2,0,0,0,0,0,0,...,1,4,1,3977,1,0,0,0,0,0
2,1,0,0,1,0,0,0,0,0,0,...,1,2,1,10788,0,0,0,0,0,0
3,2,0,0,3,0,0,0,0,0,0,...,1,2,1,14339,1,0,0,0,0,1
4,1,1,0,4,0,0,0,0,0,0,...,1,2,1,389,1,1,0,0,0,1


## 2. Data cleaning and overview

In [3]:
print("Missing values :", int(df.isna().sum().sum()))
print("Duplicate rows :", int(df.duplicated().sum()))
print("\nClass balance (before cleaning):")
print(df["phishing"].value_counts().rename({0: "legitimate", 1: "phishing"}))

# Duplicates would leak between train and test, inflating the scores, so we remove them first.
df = df.drop_duplicates().reset_index(drop=True)
print("\nShape after removing duplicates:", df.shape)
print(df["phishing"].value_counts(normalize=True).round(3).rename({0: "legitimate", 1: "phishing"}))

Missing values : 0


Duplicate rows : 1240

Class balance (before cleaning):
phishing
phishing      30647
legitimate    27998
Name: count, dtype: int64



Shape after removing duplicates: (57405, 112)
phishing
phishing      0.531
legitimate    0.469
Name: proportion, dtype: float64


**Note on `-1` values.** In this dataset `-1` means *"this part of the URL does not exist"* (for example `qty_params = -1` when the URL has no query string). It is a meaningful signal, so we keep it as it is. Tree-based models handle it naturally.

## 3. Exploratory data analysis

In [4]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
counts = df["phishing"].map({0: "Legitimate", 1: "Phishing"}).value_counts().reindex(["Legitimate", "Phishing"])
sns.barplot(x=counts.index, y=counts.values, palette=["#2a9d8f", "#e63946"], ax=ax[0])
ax[0].set_title("Class distribution"); ax[0].set_ylabel("URLs"); ax[0].set_xlabel("")
for i, v in enumerate(counts.values):
    ax[0].text(i, v, f"{v:,}", ha="center", va="bottom")

clip = df[df["length_url"] <= 200]
sns.kdeplot(data=clip, x="length_url", hue=clip["phishing"].map({0: "Legitimate", 1: "Phishing"}),
            fill=True, common_norm=False, palette=["#2a9d8f", "#e63946"], ax=ax[1])
ax[1].set_title("URL length by class (clipped at 200)"); ax[1].set_xlabel("URL length")
plt.tight_layout(); plt.savefig("images/01_class_and_length.png", dpi=150); plt.show()

In [5]:
cols = ["length_url", "qty_dot_url", "qty_slash_url", "qty_hyphen_url", "qty_equal_url", "qty_and_url", "domain_length"]
summary = df.groupby("phishing")[cols].mean().T.rename(columns={0: "Legitimate (mean)", 1: "Phishing (mean)"}).round(2)
summary

phishing,Legitimate (mean),Phishing (mean)
length_url,23.31,64.63
qty_dot_url,2.08,2.48
qty_slash_url,0.68,3.09
qty_hyphen_url,0.27,0.64
qty_equal_url,0.05,0.55
qty_and_url,0.03,0.38
domain_length,17.47,18.64


In [6]:
corr = df.corr(numeric_only=True)["phishing"].drop("phishing").sort_values()
top = pd.concat([corr.head(8), corr.tail(8)])
plt.figure(figsize=(8, 6))
sns.barplot(x=top.values, y=top.index, palette=["#2a9d8f" if v < 0 else "#e63946" for v in top.values])
plt.title("Features most correlated with 'phishing'"); plt.xlabel("Pearson correlation"); plt.ylabel("")
plt.tight_layout(); plt.savefig("images/02_correlation.png", dpi=150); plt.show()

## 4. Feature sets
The dataset has 111 features. They fall into two groups:

* **URL-only (lexical) features** - computed purely from the URL text. Instant, offline, safe (nothing is visited). This is our **main model**.
* **Network / lookup features** - need DNS, WHOIS, TLS or Google-index lookups (`time_response`, `asn_ip`, `time_domain_activation`, `qty_redirects`, `tls_ssl_certificate` ...). More informative but slow, and they require contacting the suspicious host.

We train on the URL-only set and later run an ablation to see how much the network features add.

In [7]:
TARGET = "phishing"
NETWORK = ["time_response", "domain_spf", "asn_ip", "time_domain_activation", "time_domain_expiration",
           "qty_ip_resolved", "qty_nameservers", "qty_mx_servers", "ttl_hostname", "tls_ssl_certificate",
           "qty_redirects", "url_google_index", "domain_google_index"]
ALL_FEATURES = [c for c in df.columns if c != TARGET]
LEXICAL = [c for c in ALL_FEATURES if c not in NETWORK]
print(f"URL-only features: {len(LEXICAL)}   Network features: {len(NETWORK)}   Total: {len(ALL_FEATURES)}")

X, y = df[LEXICAL], df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=SEED)
print("Train:", X_train.shape, " Test:", X_test.shape)

URL-only features: 98   Network features: 13   Total: 111
Train: (45924, 98)  Test: (11481, 98)


## 5. Train and compare four models

In [8]:
models = {
    "Logistic Regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, random_state=SEED)),
    "Decision Tree": DecisionTreeClassifier(max_depth=12, random_state=SEED),
    "Random Forest": RandomForestClassifier(n_estimators=150, min_samples_leaf=2, n_jobs=-1, random_state=SEED),
    "Gradient Boosting": HistGradientBoostingClassifier(max_iter=300, learning_rate=0.1, random_state=SEED),
}

def evaluate(model, Xtr, ytr, Xte, yte):
    t = time.time(); model.fit(Xtr, ytr); fit_s = time.time() - t
    pred = model.predict(Xte); proba = model.predict_proba(Xte)[:, 1]
    return dict(Accuracy=accuracy_score(yte, pred), Precision=precision_score(yte, pred),
                Recall=recall_score(yte, pred), F1=f1_score(yte, pred),
                ROC_AUC=roc_auc_score(yte, proba), Train_seconds=fit_s), pred, proba

rows, preds, probas = {}, {}, {}
for name, m in models.items():
    rows[name], preds[name], probas[name] = evaluate(m, X_train, y_train, X_test, y_test)
results = pd.DataFrame(rows).T.round(4).sort_values("F1", ascending=False)
results

,Accuracy,Precision,Recall,F1,ROC_AUC,Train_seconds
Gradient Boosting,0.9065,0.9258,0.8959,0.9106,0.9679,5.9693
Random Forest,0.9045,0.9170,0.9020,0.9094,0.9670,4.4523
Decision Tree,0.8932,0.9081,0.8890,0.8984,0.9447,0.5262
Logistic Regression,0.8614,0.8587,0.8847,0.8715,0.9378,3.2550


In [9]:
ax = results[["Accuracy", "Precision", "Recall", "F1", "ROC_AUC"]].plot(kind="bar", figsize=(10, 4.5), width=0.8, colormap="viridis")
ax.set_ylim(0.8, 1.0); ax.set_title("Model comparison on the held-out test set (URL-only features)")
ax.set_ylabel("Score"); plt.xticks(rotation=15); plt.legend(loc="upper center", bbox_to_anchor=(0.5, -0.18), ncol=5, fontsize=8)
plt.tight_layout(); plt.savefig("images/03_model_comparison.png", dpi=150); plt.show()

## 6. Best model: detailed evaluation

In [10]:
best_name = results.index[0]
best_model = models[best_name]
print("Best model:", best_name, "\n")
print(classification_report(y_test, preds[best_name], target_names=["Legitimate", "Phishing"], digits=4))

cm = confusion_matrix(y_test, preds[best_name])
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
ConfusionMatrixDisplay(cm, display_labels=["Legitimate", "Phishing"]).plot(ax=ax[0], cmap="Blues", colorbar=False)
ax[0].set_title(f"Confusion matrix - {best_name}")
for name, p in probas.items():
    fpr, tpr, _ = roc_curve(y_test, p)
    ax[1].plot(fpr, tpr, label=f"{name} (AUC={roc_auc_score(y_test, p):.3f})")
ax[1].plot([0, 1], [0, 1], "k--", lw=1); ax[1].set_xlabel("False positive rate"); ax[1].set_ylabel("True positive rate")
ax[1].set_title("ROC curves"); ax[1].legend(loc="lower right", fontsize=8)
plt.tight_layout(); plt.savefig("images/04_confusion_roc.png", dpi=150); plt.show()
tn, fp, fn, tp = cm.ravel()
print(f"True legit: {tn}, False alarms: {fp}, Missed phishing: {fn}, Caught phishing: {tp}")

Best model: Gradient Boosting 

              precision    recall  f1-score   support

  Legitimate     0.8862    0.9186    0.9021      5382
    Phishing     0.9258    0.8959    0.9106      6099

    accuracy                         0.9065     11481
   macro avg     0.9060    0.9073    0.9063     11481
weighted avg     0.9072    0.9065    0.9066     11481



True legit: 4944, False alarms: 438, Missed phishing: 635, Caught phishing: 5464


### 6.1 Is it stable? 5-fold cross-validation

In [11]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
cv_scores = cross_val_score(models[best_name], X, y, cv=cv, scoring="f1", n_jobs=1)
print("F1 per fold:", cv_scores.round(4))
print(f"Mean F1 = {cv_scores.mean():.4f}  (+/- {cv_scores.std():.4f})")

F1 per fold: [0.9079 0.9075 0.9047 0.9145 0.9079]
Mean F1 = 0.9085  (+/- 0.0033)


### 6.2 Where does it fail? A look at dataset bias
Overall scores can hide a problem. Let us split the data by whether the URL has a **path** (anything after the domain, e.g. `/login/index.php`).

In [12]:
has_path = (df["directory_length"] >= 0)
print("Share of phishing among URLs WITHOUT a path: %.1f%%  (n=%d)" % (100*df[~has_path][TARGET].mean(), (~has_path).sum()))
print("Share of phishing among URLs WITH a path   : %.1f%%  (n=%d)" % (100*df[has_path][TARGET].mean(), has_path.sum()))

test_idx = X_test.index
grp = pd.DataFrame({"y": y_test.values, "pred": preds[best_name], "has_path": has_path.loc[test_idx].values})
rows = []
for name, g in grp.groupby("has_path"):
    tn_, fp_, fn_, tp_ = confusion_matrix(g.y, g.pred, labels=[0, 1]).ravel()
    rows.append({"Subgroup": "URL has a path" if name else "Bare domain (no path)", "Test URLs": len(g),
                 "Phishing share": round(g.y.mean(), 3), "Accuracy": round((g.y == g.pred).mean(), 3),
                 "False-alarm rate": round(fp_ / max(fp_ + tn_, 1), 3), "Miss rate": round(fn_ / max(fn_ + tp_, 1), 3)})
subgroup = pd.DataFrame(rows).set_index("Subgroup")
subgroup

Share of phishing among URLs WITHOUT a path: 4.5%  (n=16519)
Share of phishing among URLs WITH a path   : 72.8%  (n=40886)


,Test URLs,Phishing share,Accuracy,False-alarm rate,Miss rate
Subgroup,,,,,
Bare domain (no path),3252,0.042,0.959,0.002,0.928
URL has a path,8229,0.724,0.886,0.190,0.085


**Finding.** In this dataset almost every bare-domain URL is legitimate and most URLs with a path are phishing, so a model can score well partly by learning *"short and simple = safe, long and deep = suspicious"*. This is a well-known shortcut in public phishing datasets. The consequence is visible in the false-alarm rate for URLs that have a path: **legitimate deep links (for example a GitHub repo page or a Google search link) are more likely to be flagged**. The headline accuracy is therefore optimistic for real-world traffic, and a production system would need balanced, fresher data (legitimate URLs with realistic paths) and a tuned threshold.

## 7. Which URL features matter most?

In [13]:
rf = models["Random Forest"]
imp = pd.Series(rf.feature_importances_, index=LEXICAL).sort_values(ascending=False).head(15)
plt.figure(figsize=(8, 5.5))
sns.barplot(x=imp.values, y=imp.index, color="#457b9d")
plt.title("Top 15 features (Random Forest importance)"); plt.xlabel("Importance"); plt.ylabel("")
plt.tight_layout(); plt.savefig("images/05_feature_importance.png", dpi=150); plt.show()
imp.round(4)

directory_length             0.1188
length_url                   0.0588
qty_slash_url                0.0417
file_length                  0.0407
domain_length                0.0366
qty_slash_directory          0.0357
qty_underline_directory      0.0342
qty_dot_directory            0.0336
qty_vowels_domain            0.0330
qty_dot_domain               0.0309
qty_equal_directory          0.0221
qty_percent_directory        0.0219
qty_equal_file               0.0217
qty_dot_file                 0.0196
qty_exclamation_directory    0.0182
dtype: float64

## 8. Ablation: what do the network features add?
We retrain the best model on **all 111 features** (including DNS/WHOIS/TLS lookups) to see how much accuracy we give up by staying URL-only.

In [14]:
Xf_train, Xf_test, _, _ = train_test_split(df[ALL_FEATURES], y, test_size=0.2, stratify=y, random_state=SEED)
from sklearn.base import clone
full_metrics, _, _ = evaluate(clone(models[best_name]), Xf_train, y_train, Xf_test, y_test)
ablation = pd.DataFrame({"URL-only (%d feats)" % len(LEXICAL): results.loc[best_name],
                         "All features (%d feats)" % len(ALL_FEATURES): pd.Series(full_metrics)}).T.round(4)
ablation

,Accuracy,Precision,Recall,F1,ROC_AUC,Train_seconds
URL-only (98 feats),0.9065,0.9258,0.8959,0.9106,0.9679,5.9693
All features (111 feats),0.9558,0.9591,0.9577,0.9584,0.9913,6.4353


## 9. Save the model and metrics

In [15]:
os.makedirs("models", exist_ok=True)
joblib.dump({"model": best_model, "features": LEXICAL, "name": best_name}, "models/phishing_url_model.joblib", compress=3)
print("Model size: %.1f MB" % (os.path.getsize("models/phishing_url_model.joblib") / 1e6))

out = {"best_model": best_name, "n_rows_clean": int(len(df)), "n_train": int(len(X_train)), "n_test": int(len(X_test)),
       "n_lexical_features": len(LEXICAL), "n_all_features": len(ALL_FEATURES),
       "class_counts": {"legitimate": int((y == 0).sum()), "phishing": int((y == 1).sum())},
       "test_results": results.to_dict("index"), "ablation": ablation.to_dict("index"),
       "cv_f1_mean": float(cv_scores.mean()), "cv_f1_std": float(cv_scores.std()),
       "confusion": {"tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp)},
       "top_features": imp.round(4).to_dict(),
       "subgroups": subgroup.to_dict("index")}
json.dump(out, open("results/metrics.json", "w"), indent=2)

Model size: 0.4 MB


## 10. Live prediction on raw URLs
To use the model on a real URL we need to turn the text into the same features the dataset uses. `extract_features` mirrors the dataset definitions: it splits the URL (scheme removed) into *domain / directory / file / parameters* and counts special characters in each part. A missing part is encoded as `-1`, exactly as in the data.

In [16]:
SPECIALS = {"dot": ".", "hyphen": "-", "underline": "_", "slash": "/", "questionmark": "?", "equal": "=", "at": "@",
            "and": "&", "exclamation": "!", "space": " ", "tilde": "~", "comma": ",", "plus": "+",
            "asterisk": "*", "hashtag": "#", "dollar": "$", "percent": "%"}
SHORTENERS = ("bit.ly", "goo.gl", "tinyurl.com", "t.co", "ow.ly", "is.gd", "buff.ly", "rebrand.ly", "cutt.ly", "shorturl.at")
TLDS = (".com", ".org", ".net", ".edu", ".gov", ".io", ".co", ".info", ".biz", ".in", ".uk", ".de", ".ru", ".cn", ".xyz", ".top", ".tk")

def _counts(s, part):
    return {f"qty_{k}_{part}": s.count(ch) for k, ch in SPECIALS.items()}

def extract_features(url: str) -> dict:
    u = re.sub(r"^[a-zA-Z][a-zA-Z0-9+.-]*://", "", url.strip())        # dataset URLs have no scheme
    host_end = min([i for i in (u.find("/"), u.find("?"), u.find("#")) if i != -1] or [len(u)])
    domain, rest = u[:host_end], u[host_end:]
    path, _, params = rest.partition("?")
    f = {}
    f.update(_counts(u, "url"))
    f["qty_tld_url"] = sum(u.lower().count(t) for t in TLDS)
    f["length_url"] = len(u)
    f.update(_counts(domain, "domain"))
    f["qty_vowels_domain"] = sum(domain.lower().count(v) for v in "aeiou")
    f["domain_length"] = len(domain)
    f["domain_in_ip"] = int(bool(re.fullmatch(r"\d{1,3}(\.\d{1,3}){3}(:\d+)?", domain)))
    f["server_client_domain"] = int("server" in domain.lower() or "client" in domain.lower())
    if "/" in path:
        directory, _, file = path.rpartition("/"); directory += "/"
    else:
        directory = file = None
    for part, s in (("directory", directory), ("file", file)):
        f.update(_counts(s, part) if s is not None else {f"qty_{k}_{part}": -1 for k in SPECIALS})
        f[f"{part}_length"] = len(s) if s is not None else -1
    if "?" in rest:
        f.update(_counts(params, "params")); f["params_length"] = len(params)
        f["tld_present_params"] = int(any(t in params.lower() for t in TLDS))
        f["qty_params"] = len([p for p in params.split("&") if p])
    else:
        f.update({f"qty_{k}_params": -1 for k in SPECIALS}); f["params_length"] = -1
        f["tld_present_params"] = -1; f["qty_params"] = -1
    f["email_in_url"] = int(bool(re.search(r"[\w.+-]+@[\w-]+\.[\w.-]+", u)))
    f["url_shortened"] = int(any(domain.lower().endswith(s) for s in SHORTENERS))
    return f

bundle = joblib.load("models/phishing_url_model.joblib")

def predict_url(url: str):
    row = pd.DataFrame([extract_features(url)]).reindex(columns=bundle["features"], fill_value=0)
    p = float(bundle["model"].predict_proba(row)[0, 1])
    return ("PHISHING" if p >= 0.5 else "legitimate"), p

missing = set(LEXICAL) - set(extract_features("example.com/a?b=1"))
print("Features not produced by extractor:", missing or "none")

Features not produced by extractor: none


In [17]:
samples = [
    "https://www.wikipedia.org/",
    "https://github.com/scikit-learn/scikit-learn",
    "https://www.google.com/search?q=machine+learning",
    "http://192.168.44.12/secure/login.php?user=admin&redirect=http://paypal.com.verify-account.info",
    "http://secure-paypal-login.verify-account-update.com/signin/confirm/identity/index.php?id=29384&session=ab12cd",
    "http://bit.ly/3xFr9Qa",
]
table = pd.DataFrame([(u[:85] + ("..." if len(u) > 85 else ""), *predict_url(u)) for u in samples],
                     columns=["URL", "Verdict", "P(phishing)"])
table["P(phishing)"] = table["P(phishing)"].round(3)
table

,URL,Verdict,P(phishing)
0,https://www.wikipedia.org/,legitimate,0.171
1,https://github.com/scikit-learn/scikit-learn,PHISHING,0.917
2,https://www.google.com/search?q=machine+learning,PHISHING,0.915
3,http://192.168.44.12/secure/login.php?user=adm...,PHISHING,0.993
4,http://secure-paypal-login.verify-account-upda...,PHISHING,0.999
5,http://bit.ly/3xFr9Qa,PHISHING,0.947


**Reading these results honestly.** The obviously malicious examples (IP-address host, brand-in-subdomain tricks, long credential-style path) are caught with very high confidence, and a bare legitimate domain passes. But the **GitHub and Google links are false alarms**, a direct effect of the path-length bias shown in section 6.2. We keep them in the demo on purpose: a model's weaknesses should be visible, not hidden.

## 11. Conclusion, limitations and future work
* A model that only reads the URL text separates phishing from legitimate links well on this dataset (about 91% F1), and tree ensembles clearly beat the linear baseline because the relationships between URL features are non-linear.
* Adding DNS/WHOIS/TLS lookups improves results further (ablation in section 8), but at the price of speed and of contacting the suspicious host.
* **Limitations:** the dataset has a strong path/length bias (section 6.2), so legitimate deep links can be flagged. The dataset is a snapshot from one period, and attackers change tricks over time, so the model needs retraining on fresh data. Lexical features can be evaded by a careful attacker who crafts a "clean-looking" URL. The raw-URL extractor is a close re-implementation of the dataset definitions, not the original code, so small differences in a few features are possible. It is a **screening aid, not a replacement for a security gateway**.
* **Future work:** add character n-gram / deep-learning URL models, a browser extension or Flask API around `predict_url`, periodic retraining, and probability calibration with a tuned decision threshold.